- D-Fire YOLO26x | Google Colab | 1 GPU T4
  - Nguồn dữ liệu: folder `D-Fire` từ `D-Fire-train-ready.zip` trong Google Drive
  - Stage train: `D-Fire-clean`; symlink ảnh, label đã clip độc lập từng trục, bỏ box zero-area
  - Split khóa: `train`, `valid`, `test`; chọn model bằng `valid`; đánh giá cuối bằng `test`
  - Protocol: 800 px, 20 epoch đầy đủ, seed 20260707, effective batch 32
  - LR recipe: native Ultralytics 8.4.90 (`cos_lr=False`, warmup 3 epoch)
  - Augmentation: trainer recipe mặc định Ultralytics; không custom override
  - Output/checkpoint: Google Drive; tự resume từ `last.pt`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import subprocess

ARCHIVE_PATH = Path('/content/drive/MyDrive/CV_Lab/D-Fire-train-ready.zip')
SOURCE_DATA_ROOT = Path('/content/D-Fire')
DATA_ROOT = Path('/content/D-Fire-clean')
OUTPUT_ROOT = Path('/content/drive/MyDrive/cv-inference-lab/runs')
RUN_NAME = 'dfire_yolo26x_res800_b8'
SEED = 20260707
EPOCHS = 20
RESOLUTION = 800
BATCH = 8
NOMINAL_BATCH = 32
RUN_DIR = OUTPUT_ROOT / RUN_NAME
if not SOURCE_DATA_ROOT.is_dir():
    assert ARCHIVE_PATH.is_file(), f'Không thấy archive: {ARCHIVE_PATH}'
    subprocess.run(['unzip', '-q', str(ARCHIVE_PATH), '-d', '/content'], check=True)
assert SOURCE_DATA_ROOT.is_dir(), f'Không thấy folder dataset: {SOURCE_DATA_ROOT}'
print({'source_data_root': str(SOURCE_DATA_ROOT), 'clean_data_root': str(DATA_ROOT), 'run_dir': str(RUN_DIR)})


In [ ]:
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.90'], check=True)

import torch
import ultralytics

assert torch.cuda.is_available(), 'Colab runtime chưa bật GPU'
devices = [torch.cuda.get_device_name(index) for index in range(torch.cuda.device_count())]
capabilities = [torch.cuda.get_device_capability(index) for index in range(torch.cuda.device_count())]
print({'torch': torch.__version__, 'cuda': torch.version.cuda, 'ultralytics': ultralytics.__version__, 'devices': devices, 'capabilities': capabilities})
assert len(devices) == 1 and 'T4' in devices[0], f'Cần Colab 1 GPU T4, hiện có {devices}'
assert capabilities[0] >= (7, 0), f'GPU không được torch {torch.__version__} hỗ trợ: {capabilities}'


In [ ]:
import math
import os
import shutil

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png'}
EXPECTED_COUNTS = {'train': 15500, 'valid': 1721, 'test': 4306}
VALID_CLASS_IDS = {0, 1}
issues = []
summary = {}
split_images = {}
shutil.rmtree(DATA_ROOT, ignore_errors=True)
DATA_ROOT.mkdir(parents=True)
for split, expected_count in EXPECTED_COUNTS.items():
    source_images = SOURCE_DATA_ROOT / split / 'images'
    source_labels = SOURCE_DATA_ROOT / split / 'labels'
    assert source_images.is_dir(), f'Thiếu: {source_images}'
    assert source_labels.is_dir(), f'Thiếu: {source_labels}'
    images = sorted(path for path in source_images.iterdir() if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)
    labels = sorted(path for path in source_labels.iterdir() if path.is_file() and path.suffix.lower() == '.txt')
    image_by_stem = {path.stem: path for path in images}
    label_stems = [path.stem for path in labels]
    assert len(images) == expected_count, f'{split}: cần {expected_count} ảnh, thấy {len(images)}'
    assert len(image_by_stem) == len(images), f'{split}: trùng tên stem ảnh'
    assert len(label_stems) == len(set(label_stems)), f'{split}: trùng tên stem label'
    orphan_labels = sorted(set(label_stems) - set(image_by_stem))
    assert not orphan_labels, f'{split}: label không có ảnh: {orphan_labels[:10]}'
    clean_split = DATA_ROOT / split
    clean_labels = clean_split / 'labels'
    clean_split.mkdir()
    clean_labels.mkdir()
    os.symlink(source_images, clean_split / 'images', target_is_directory=True)
    input_boxes = 0
    kept_boxes = 0
    clipped_boxes = 0
    dropped_boxes = 0
    invalid = []
    for label_path in labels:
        cleaned_lines = []
        for line_number, raw_line in enumerate(label_path.read_text(encoding='utf-8').splitlines(), 1):
            line = raw_line.strip()
            if not line:
                continue
            input_boxes += 1
            parts = line.split()
            try:
                class_id = int(parts[0])
                x_center, y_center, box_width, box_height = [float(value) for value in parts[1:]]
                syntax_valid = len(parts) == 5 and class_id in VALID_CLASS_IDS and all(math.isfinite(value) for value in (x_center, y_center, box_width, box_height))
            except (ValueError, IndexError):
                syntax_valid = False
            if not syntax_valid:
                invalid.append(f'{label_path.name}:{line_number}')
                continue
            x1 = x_center - box_width / 2.0
            y1 = y_center - box_height / 2.0
            x2 = x_center + box_width / 2.0
            y2 = y_center + box_height / 2.0
            clipped = (max(0.0, min(1.0, x1)), max(0.0, min(1.0, y1)), max(0.0, min(1.0, x2)), max(0.0, min(1.0, y2)))
            original = (x_center, y_center, box_width, box_height)
            if clipped[2] <= clipped[0] or clipped[3] <= clipped[1]:
                dropped_boxes += 1
                issues.append({'split': split, 'image_path': str(image_by_stem[label_path.stem]), 'label': label_path.name, 'line': line_number, 'action': 'drop_zero_area', 'original': original, 'fixed': None})
                continue
            fixed = ((clipped[0] + clipped[2]) / 2.0, (clipped[1] + clipped[3]) / 2.0, clipped[2] - clipped[0], clipped[3] - clipped[1])
            if clipped != (x1, y1, x2, y2):
                clipped_boxes += 1
                issues.append({'split': split, 'image_path': str(image_by_stem[label_path.stem]), 'label': label_path.name, 'line': line_number, 'action': 'clip_edges', 'original': original, 'fixed': fixed})
                line = f'{class_id} ' + ' '.join(f'{value:.12g}' for value in fixed)
            cleaned_lines.append(line)
            kept_boxes += 1
        target = clean_labels / label_path.name
        target.write_text(('\n'.join(cleaned_lines) + '\n') if cleaned_lines else '', encoding='utf-8')
    assert not invalid, f'{split}: label sai cú pháp/class/non-finite: {invalid[:10]}'
    missing_labels = sorted(set(image_by_stem) - set(label_stems))
    split_images[split] = set(image_by_stem)
    summary[split] = {'images': len(images), 'labels': len(labels), 'missing_labels_as_background': len(missing_labels), 'input_boxes': input_boxes, 'kept_boxes': kept_boxes, 'clipped_boxes': clipped_boxes, 'dropped_boxes': dropped_boxes}
for left, right in (('train', 'valid'), ('train', 'test'), ('valid', 'test')):
    overlap = sorted(split_images[left] & split_images[right])
    assert not overlap, f'Rò split {left}/{right}: {overlap[:10]}'
print(summary)
print({'affected_files': len({(item['split'], item['label']) for item in issues}), 'affected_boxes': len(issues), 'status': 'PASS'})


In [ ]:
from collections import defaultdict
from IPython.display import display
import ipywidgets as widgets
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from PIL import Image

issues_by_file = defaultdict(list)
for item in issues:
    issues_by_file[(item['split'], item['image_path'], item['label'])].append(item)
affected_files = sorted(issues_by_file)
for index, (split, image_path, label_name) in enumerate(affected_files, 1):
    locations = ', '.join(f"{item['line']}:{item['action']}" for item in issues_by_file[(split, image_path, label_name)])
    print(f'{index:04d} {split}/{label_name} -> {locations}')
def read_boxes(path):
    boxes = []
    for line in path.read_text(encoding='utf-8').splitlines():
        parts = line.split()
        if len(parts) == 5:
            boxes.append((int(parts[0]), *[float(value) for value in parts[1:]]))
    return boxes
def draw_box(axis, box, image_width, image_height, color, linestyle='-', linewidth=1.5):
    x_center, y_center, box_width, box_height = box
    x1 = (x_center - box_width / 2.0) * image_width
    y1 = (y_center - box_height / 2.0) * image_height
    axis.add_patch(Rectangle((x1, y1), box_width * image_width, box_height * image_height, fill=False, edgecolor=color, linestyle=linestyle, linewidth=linewidth, clip_on=False))
def show_issue_file(index):
    split, image_path, label_name = affected_files[index]
    image = Image.open(image_path).convert('RGB')
    image_width, image_height = image.size
    ratio = image_width / image_height
    figure_size = (7.0, max(2.8, 7.0 / ratio)) if ratio >= 1.0 else (max(2.8, 7.0 * ratio), 7.0)
    figure, axis = plt.subplots(figsize=figure_size, dpi=100)
    axis.imshow(image, extent=(0, image_width, image_height, 0))
    for _, x_center, y_center, box_width, box_height in read_boxes(DATA_ROOT / split / 'labels' / label_name):
        draw_box(axis, (x_center, y_center, box_width, box_height), image_width, image_height, '#00ff66')
    for item in issues_by_file[(split, image_path, label_name)]:
        draw_box(axis, item['original'], image_width, image_height, '#ff3030', '--', 2.0)
        if item['fixed'] is None:
            axis.plot(item['original'][0] * image_width, item['original'][1] * image_height, marker='x', color='#ff3030', markersize=9, markeredgewidth=2)
    padding = max(image_width, image_height) * 0.025
    axis.set_xlim(-padding, image_width + padding)
    axis.set_ylim(image_height + padding, -padding)
    axis.set_aspect('equal', adjustable='box')
    axis.set_title(f'{index + 1}/{len(affected_files)} | {split}/{label_name} | {image_width}x{image_height}', fontsize=9)
    axis.legend(handles=[Line2D([0], [0], color='#ff3030', linestyle='--', label='gốc/lỗi'), Line2D([0], [0], color='#00ff66', label='sau sửa')], loc='upper right', fontsize=8)
    axis.set_xlabel('pixel x')
    axis.set_ylabel('pixel y')
    plt.show()
if affected_files:
    slider = widgets.IntSlider(value=0, min=0, max=len(affected_files) - 1, step=1, description='file', continuous_update=False, layout=widgets.Layout(width='650px'))
    output = widgets.interactive_output(show_issue_file, {'index': slider})
    display(widgets.VBox([slider, output]))
else:
    print('Không có label cần sửa')


In [ ]:
import yaml

data_yaml = DATA_ROOT / 'data.yaml'
data_yaml.write_text(yaml.safe_dump({'path': str(DATA_ROOT), 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images', 'names': {0: 'smoke', 1: 'fire'}}, sort_keys=False), encoding='utf-8')
print(data_yaml.read_text(encoding='utf-8'))


In [ ]:
import json

PROTOCOL = {'schema': 2, 'framework': 'ultralytics-8.4.90', 'model': 'yolo26x', 'dataset': 'D-Fire-fixed-split-clip-edges-drop-degenerate', 'resolution': RESOLUTION, 'epochs': EPOCHS, 'seed': SEED, 'batch': BATCH, 'nominal_batch': NOMINAL_BATCH, 'early_stopping': False, 'augmentation_policy': 'framework-default-no-user-overrides'}
protocol_path = RUN_DIR / 'training_protocol.json'
last_path = RUN_DIR / 'weights' / 'last.pt'
if RUN_DIR.exists():
    assert protocol_path.is_file(), f'Run cũ thiếu protocol: {protocol_path}'
    saved_protocol = json.loads(protocol_path.read_text(encoding='utf-8'))
    assert saved_protocol == PROTOCOL, {'expected': PROTOCOL, 'actual': saved_protocol}
else:
    RUN_DIR.mkdir(parents=True)
    protocol_path.write_text(json.dumps(PROTOCOL, indent=2, sort_keys=True), encoding='utf-8')
results_path = RUN_DIR / 'results.csv'
completed_epochs = max(sum(1 for _ in results_path.open(encoding='utf-8')) - 1, 0) if results_path.is_file() else 0
run_complete = completed_epochs >= EPOCHS and (RUN_DIR / 'weights' / 'best.pt').is_file()
resume_path = last_path if last_path.is_file() and not run_complete else None
checkpoint_epoch = None
if resume_path:
    checkpoint = torch.load(resume_path, map_location='cpu', weights_only=False)
    args = checkpoint.get('train_args', {})
    expected = {'imgsz': RESOLUTION, 'epochs': EPOCHS, 'seed': SEED, 'batch': BATCH, 'nbs': NOMINAL_BATCH, 'patience': 0}
    mismatches = {key: {'expected': value, 'actual': args.get(key)} for key, value in expected.items() if args.get(key) != value}
    assert not mismatches, mismatches
    checkpoint_epoch = int(checkpoint['epoch'])
    assert checkpoint_epoch >= 0, f'Checkpoint không còn full-state để resume: epoch={checkpoint_epoch}'
print({'resume': str(resume_path) if resume_path else None, 'checkpoint_epoch_zero_based': checkpoint_epoch, 'completed_epochs_from_results': completed_epochs, 'complete': run_complete})


In [ ]:
from ultralytics import YOLO

if run_complete:
    print(f'Không train lại: đã đủ {EPOCHS} epoch')
elif resume_path:
    model = YOLO(str(resume_path))
    model.train(resume=str(resume_path), data=str(data_yaml), device=0, workers=2)
else:
    model = YOLO('yolo26x.pt')
    model.train(data=str(data_yaml), project=str(OUTPUT_ROOT), name=RUN_NAME, exist_ok=True, epochs=EPOCHS, imgsz=RESOLUTION, batch=BATCH, nbs=NOMINAL_BATCH, device=0, workers=2, seed=SEED, patience=0, save_period=1)


In [ ]:
best_path = RUN_DIR / 'weights' / 'best.pt'
last_path = RUN_DIR / 'weights' / 'last.pt'
assert best_path.is_file(), best_path
assert last_path.is_file(), last_path
evaluator = YOLO(str(best_path))
test_metrics = evaluator.val(data=str(data_yaml), split='test', imgsz=RESOLUTION, batch=BATCH, device=0, workers=2, augment=False, project=str(RUN_DIR), name='test_evaluation', exist_ok=True, plots=True)
print(test_metrics.results_dict)


In [ ]:
for path in sorted(RUN_DIR.rglob('*')):
    if path.is_file() and path.suffix.lower() in {'.pt', '.csv', '.json'}:
        print(path, path.stat().st_size)